# Remote Cleanup 03: Manual `atexit` Check with Kernel Restart

This notebook is for the orderly shutdown path.

1. Create live remote resources from a device discovered through the normal `import pynq` / `Device.devices` path, where default `auto_cleanup=True` applies.
2. Save the remote IDs to a small state file.
3. Check the board logs while the resources are live.
4. Restart the notebook kernel.
5. Probe the old IDs from the new kernel using `RemoteDevice(auto_cleanup=False)`.

If the old IDs are already invalid after restart, that is the expected result for an orderly kernel shutdown where `atexit` cleanup ran successfully.

In [ ]:
import json
import os
from pathlib import Path

import grpc
import numpy as np

REMOTE_IP = os.environ.get("PYNQ_REMOTE_DEVICES", "192.168.2.197").split(",")[0].strip()
USE_PROBED_DEVICE_FOR_AUTO_CLEANUP = True
OVERLAY_NAME = "resizer.xsa"
STATE_FILE = Path("/tmp/pynq_remote_atexit_state.json")

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "tests" / OVERLAY_NAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not find tests/{OVERLAY_NAME} from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
OVERLAY_PATH = REPO_ROOT / "tests" / OVERLAY_NAME

os.environ["PYNQ_REMOTE_DEVICES"] = REMOTE_IP

import pynq
from pynq import GPIO, Overlay
from pynq.pl_server.device import Device
from pynq.pl_server.remote_device import RemoteDevice
from pynq.remote import buffer_pb2, gpio_pb2, mmio_pb2

def reset_pynq_device_probe():
    if hasattr(Device, "_active_device"):
        delattr(Device, "_active_device")
    if hasattr(Device, "_devices"):
        delattr(Device, "_devices")

def get_device(auto_cleanup: bool = True):
    if auto_cleanup and USE_PROBED_DEVICE_FOR_AUTO_CLEANUP:
        reset_pynq_device_probe()
        devices = [d for d in Device.devices if isinstance(d, RemoteDevice)]
        if not devices:
            raise RuntimeError("No probed RemoteDevice found. Check PYNQ_REMOTE_DEVICES and connectivity.")
        return devices[0]
    return RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=auto_cleanup)

print(f"REMOTE_IP={REMOTE_IP}")
print(f"USE_PROBED_DEVICE_FOR_AUTO_CLEANUP={USE_PROBED_DEVICE_FOR_AUTO_CLEANUP}")
print(f"OVERLAY_PATH={OVERLAY_PATH}")
print(f"STATE_FILE={STATE_FILE}")

In [ ]:
device = get_device(auto_cleanup=True)
overlay = Overlay(str(OVERLAY_PATH), device=device)
base_addr = overlay.ip_dict["resize_accel_0"]["phys_addr"]

remote_mmio = device.mmap(base_addr, 0x1000)
remote_mmio.read(0)

remote_buffer = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
remote_buffer[:] = np.arange(16, dtype=np.uint32)
remote_buffer.flush()

remote_gpio = None
gpio_id = None
gpio_path = None
base_path = GPIO.get_gpio_base_path(device=device)
npins = GPIO.get_gpio_npins(device=device)
if base_path and npins:
    gpio_pin = GPIO.get_gpio_pin(0, device=device)
    gpio_path = f"/sys/class/gpio/gpio{gpio_pin}"
    if not device.exists_file(gpio_path).exists:
        remote_gpio = GPIO(gpio_pin, "in", device=device)
        remote_gpio.read()
        gpio_id = remote_gpio._gpio_id
    else:
        print(f"Skipping GPIO because {gpio_path} is already exported.")
else:
    print("Skipping GPIO because Linux sysfs GPIO is not available.")

state = {
    "mmio_id": remote_mmio.mmio_id,
    "buffer_id": remote_buffer.buffer_id,
    "gpio_id": gpio_id,
    "gpio_path": gpio_path,
}
STATE_FILE.write_text(json.dumps(state, indent=2))
print(state)
input("Check the board logs for the live resources, then restart the notebook kernel before running the next cell.")

In [ ]:
state = json.loads(STATE_FILE.read_text())
print(state)

probe_device = RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=False)

def id_is_live():
    results = {}
    try:
        probe_device._stub["mmio"].read(
            mmio_pb2.ReadRequest(mmio_id=state["mmio_id"], offset=0, length=4, word_order="little")
        )
        results["mmio"] = True
    except (grpc.RpcError, RuntimeError):
        results["mmio"] = False
    try:
        probe_device._stub["buffer"].physical_address(
            buffer_pb2.AddressRequest(buffer_id=state["buffer_id"])
        )
        results["buffer"] = True
    except (grpc.RpcError, RuntimeError):
        results["buffer"] = False
    if state["gpio_id"] is not None:
        try:
            probe_device._stub["gpio"].read(gpio_pb2.GpioReadRequest(gpio_id=state["gpio_id"]))
            results["gpio"] = True
        except (grpc.RpcError, RuntimeError):
            results["gpio"] = False
    return results

results = id_is_live()
print("After restart:", results)
for resource_name, is_live in results.items():
    if is_live:
        print(f"{resource_name} is still live after restart. That suggests the previous kernel exit did not clean it up.")
    else:
        print(f"{resource_name} is already invalid after restart as expected for orderly atexit cleanup.")
if state["gpio_path"] is not None:
    print("GPIO path exists after restart:", probe_device.exists_file(state["gpio_path"]).exists)